In [ ]:
from qgis.core import QgsField, QgsProject, edit
from PyQt5.QtCore import QVariant

# ── Config ─────────────────────────────────────────────────────────────────
# Change this to match the exact name of your zonal histogram output layer
LAYER_NAME = "Output zones"

CLASS_MAP = {
    "px_green_res": "HISTO_NODATA",
    "px_parkland":  "HISTO_1",
    "px_urban":     "HISTO_2",
    "px_water":     "HISTO_3",
}
# ───────────────────────────────────────────────────────────────────────────

layer = QgsProject.instance().mapLayersByName(LAYER_NAME)[0]

new_fields = (
    list(CLASS_MAP.keys()) +
    ["px_total", "pct_green_res", "pct_parkland", "pct_urban", "pct_water"]
)

with edit(layer):
    # Add all new fields
    for name in new_fields:
        if layer.fields().indexOf(name) == -1:
            layer.addAttribute(QgsField(name, QVariant.Double))
    layer.updateFields()

    for feature in layer.getFeatures():
        # Raw counts
        counts = {k: (feature[v] or 0) for k, v in CLASS_MAP.items()}
        total = sum(counts.values())

        pcts = {
            f"pct_{k.replace('px_', '')}": (
                round(counts[k] / total * 100, 1) if total > 0 else 0
            )
            for k in counts
        }

        layer.changeAttributeValue(feature.id(), layer.fields().indexOf("px_total"),    total)
        for fname, val in {**counts, **pcts}.items():
            layer.changeAttributeValue(feature.id(), layer.fields().indexOf(fname), val)

print("Done — fields added and populated.")